# Logit Calibration and Last-Layer Diagnostics for Merging (n=8)

This notebook is designed to validate the failure mode you observed after naive merging:
- `best@8` is relatively preserved,
- while `mean@8` / `worst@8` drop sharply.

The analysis focuses on:
1. **Logit-level uncertainty and calibration** (`entropy`, `confidence`, `ECE`, `Brier`).
2. **Prompt-level consistency** across 8 samples (`mixed outcome rate`, `confidence/entropy gaps`).
3. **Last-layer drift** near logits (`lm_head`, `final_norm`) relative to a reference model.


## Metric Design (Core Hypothesis)

If naive merging hurts consistency, expected signatures are:

- **Higher entropy** and lower margin on incorrect samples.
- **Weaker calibration**: confidence no longer tracks correctness well.
- **Worse prompt stability**: more prompts have mixed outcomes across `n=8`.
- **Inverted confidence ordering**: wrong samples can appear more confident than correct ones.
- **Last-layer drift**: `lm_head` / `final_norm` move in directions that degrade logit reliability.

Main metrics in this notebook:

- Token-level: `entropy_mean`, `top1_prob_mean`, `margin_mean`, `nll_mean`, `ppl`.
- Calibration: `ECE`, `Brier`, reliability bins.
- Consistency: `acc/mean`, `acc/best`, `acc/worst`, `mixed_prompt_rate`, `confidence_inversion_rate`.
- Last-layer: delta norm ratio, cosine similarity, active sign-flip ratio.


In [ ]:
from __future__ import annotations

import gc
import hashlib
import json
import math
import random
from dataclasses import dataclass, field
from pathlib import Path
from typing import Dict, Iterable, List, Optional, Sequence, Tuple

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
from tqdm.auto import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer

# --------------------------------------------------------------------------------------
# Global plotting style and reproducibility controls.
# --------------------------------------------------------------------------------------
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

sns.set_theme(style="whitegrid", context="talk")

# All analysis artifacts (tables and figures) are written under this directory.
ARTIFACT_DIR = Path("merging_analysis/artifacts/logit_calibration")
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)


@dataclass(frozen=True)
class ModelRunConfig:
    """Configuration for one model run to analyze.

    Args:
        display_name: User-facing model label shown in tables and plots.
        model_path: HF model id or local checkpoint directory.
        output_jsonl: JSONL file with generation outputs that includes at least
            `input` and `output`. If available, `acc` is used for calibration.
    """

    display_name: str
    model_path: Path | str
    output_jsonl: Optional[Path] = None


@dataclass(frozen=True)
class AnalysisConfig:
    """Top-level execution knobs for this notebook.

    Args:
        model_runs: Model/output pairs that participate in analysis.
        max_prompts_per_model: Optional prompt cap per model for faster iteration.
        max_samples_per_prompt: Keep at most this many samples per prompt id.
        max_examples_for_teacher_forcing: Optional row cap per model for logit scoring.
        max_sequence_length: Maximum tokenized prompt+response length accepted.
        teacher_forcing_batch_size: Batch size for forward pass (use 1 for long outputs).
        calibration_bins: Number of bins for reliability and ECE.
        tau_rms_factor: Threshold scale used for active sign-flip in last-layer diagnostics.
        device: Explicit device override. If None, auto-select cuda when available.
    """

    model_runs: Sequence[ModelRunConfig] = field(default_factory=list)
    max_prompts_per_model: Optional[int] = 120
    max_samples_per_prompt: int = 8
    max_examples_for_teacher_forcing: Optional[int] = 640
    max_sequence_length: int = 4096
    teacher_forcing_batch_size: int = 1
    calibration_bins: int = 10
    tau_rms_factor: float = 0.1
    device: Optional[str] = None


# --------------------------------------------------------------------------------------
# Default local setup in this environment.
# Replace or extend these entries with your full evaluation outputs for:
# Base / Math / IFEval / Math->IFEval / Merge-weighted / Merge-uniform.
# --------------------------------------------------------------------------------------
CONFIG = AnalysisConfig(
    model_runs=[
        ModelRunConfig(
            display_name="Math-stage2-step130",
            model_path=Path("/mnt/ddn/vuvlm/geeho/nemotron_cascade_output/Qwen3-1.7B-math/stage2/global_step_40/actor/huggingface"),
            output_jsonl=Path("/mnt/ddn/vuvlm/geeho/nemotron_cascade_output/Qwen3-1.7B-math/stage2/validation_outputs/130.jsonl"),
        ),
        ModelRunConfig(
            display_name="IfMath-stage2-step10",
            model_path=Path("/mnt/ddn/vuvlm/geeho/nemotron_cascade_output/Qwen3-1.7B-if-math/stage1/global_step_25/actor/huggingface"),
            output_jsonl=Path("/mnt/ddn/vuvlm/geeho/nemotron_cascade_output/Qwen3-1.7B-if-math/stage2/validation_outputs/10.jsonl"),
        ),
        # Example (uncomment and set real files):
        # ModelRunConfig(
        #     display_name="Base",
        #     model_path="Qwen/Qwen3-1.7B",
        #     output_jsonl=Path("/path/to/base_eval_outputs.jsonl"),
        # ),
        # ModelRunConfig(
        #     display_name="Merge-uniform",
        #     model_path=Path("/mnt/ddn/vuvlm/geeho/nemotron_cascade_output/Qwen3-1.7B-naive-merge/uniform"),
        #     output_jsonl=Path("/path/to/merge_uniform_eval_outputs.jsonl"),
        # ),
    ],
)

print(f"Artifacts will be written to: {ARTIFACT_DIR}")
print(f"Configured model runs: {len(CONFIG.model_runs)}")
for run in CONFIG.model_runs:
    print(f"- {run.display_name}")


In [ ]:
def stable_prompt_id(prompt_text: str) -> str:
    """Create a deterministic prompt id from prompt text.

    Args:
        prompt_text: Full prompt string.

    Returns:
        Short SHA-1 digest for stable grouping across files.
    """

    digest = hashlib.sha1(prompt_text.encode("utf-8")).hexdigest()
    return digest[:16]


def normalize_accuracy_value(raw_value) -> float:
    """Convert mixed accuracy representations into a numeric {0, 1} style value.

    Args:
        raw_value: Value from JSONL (`acc`, `score`, etc.).

    Returns:
        Float accuracy value. Returns NaN if conversion is impossible.
    """

    if raw_value is None:
        return float("nan")
    if isinstance(raw_value, bool):
        return float(raw_value)
    if isinstance(raw_value, (int, float)):
        return float(raw_value)
    try:
        lowered = str(raw_value).strip().lower()
        if lowered in {"true", "t", "yes"}:
            return 1.0
        if lowered in {"false", "f", "no"}:
            return 0.0
        return float(lowered)
    except Exception:
        return float("nan")


def load_model_output_jsonl(
    run_config: ModelRunConfig,
    max_prompts: Optional[int] = None,
    max_samples_per_prompt: int = 8,
) -> pd.DataFrame:
    """Load one model's JSONL outputs into a tidy dataframe.

    Expected JSONL fields per line:
    - required: `input`, `output`
    - optional: `acc`, `score`, `reward`, `pred`, `gts`, `step`

    Args:
        run_config: Model/output configuration.
        max_prompts: Optional cap on unique prompts for faster iteration.
        max_samples_per_prompt: Per-prompt sample cap (aligns with n=8 setup).

    Returns:
        DataFrame with one row per sampled response.
    """

    if run_config.output_jsonl is None:
        raise ValueError(f"output_jsonl is not set for {run_config.display_name}")

    output_path = Path(run_config.output_jsonl)
    if not output_path.exists():
        raise FileNotFoundError(f"Output JSONL does not exist: {output_path}")

    records: List[Dict[str, object]] = []
    prompt_seen_order: Dict[str, int] = {}
    prompt_sample_counts: Dict[str, int] = {}

    with output_path.open("r", encoding="utf-8") as handle:
        for line_idx, line in enumerate(handle):
            line = line.strip()
            if not line:
                continue

            item = json.loads(line)

            prompt_text = item.get("input", item.get("prompt", ""))
            response_text = item.get("output", item.get("response", ""))
            prompt_id = stable_prompt_id(prompt_text)

            if prompt_id not in prompt_seen_order:
                prompt_seen_order[prompt_id] = len(prompt_seen_order)

            if max_prompts is not None and prompt_seen_order[prompt_id] >= max_prompts:
                continue

            prompt_sample_counts.setdefault(prompt_id, 0)
            if prompt_sample_counts[prompt_id] >= max_samples_per_prompt:
                continue

            sample_idx = prompt_sample_counts[prompt_id]
            prompt_sample_counts[prompt_id] += 1

            acc_value = normalize_accuracy_value(item.get("acc", item.get("score")))

            records.append(
                {
                    "model_name": run_config.display_name,
                    "model_path": str(run_config.model_path),
                    "source_jsonl": str(output_path),
                    "line_idx": line_idx,
                    "prompt_id": prompt_id,
                    "sample_idx": sample_idx,
                    "prompt": prompt_text,
                    "response": response_text,
                    "acc": acc_value,
                    "score": item.get("score", float("nan")),
                    "reward": item.get("reward", float("nan")),
                    "pred": item.get("pred", None),
                    "gts": item.get("gts", None),
                    "step": item.get("step", None),
                    "extraction_failed": item.get("extraction_failed", None),
                }
            )

    df = pd.DataFrame(records)
    if df.empty:
        raise ValueError(f"No records were loaded from {output_path}")

    # Row id is the primary key for merging teacher-forced metrics back later.
    df["row_id"] = df.apply(
        lambda r: f"{r['model_name']}::{r['prompt_id']}::{int(r['sample_idx'])}::{int(r['line_idx'])}",
        axis=1,
    )

    return df


def load_all_model_outputs(config: AnalysisConfig) -> pd.DataFrame:
    """Load all configured JSONL outputs and concatenate them.

    Args:
        config: Global analysis configuration.

    Returns:
        Combined dataframe sorted by model/prompt/sample index.
    """

    model_frames: List[pd.DataFrame] = []

    for run in config.model_runs:
        if run.output_jsonl is None:
            print(f"[SKIP] {run.display_name}: output_jsonl is None")
            continue

        try:
            frame = load_model_output_jsonl(
                run_config=run,
                max_prompts=config.max_prompts_per_model,
                max_samples_per_prompt=config.max_samples_per_prompt,
            )
            model_frames.append(frame)
            print(f"[OK] Loaded {len(frame):,} rows for {run.display_name}")
        except Exception as exc:
            print(f"[WARN] Failed to load {run.display_name}: {exc}")

    if not model_frames:
        raise ValueError("No output JSONL files were loaded. Please update CONFIG.model_runs.")

    merged_df = pd.concat(model_frames, ignore_index=True)
    merged_df = merged_df.sort_values(["model_name", "prompt_id", "sample_idx"]).reset_index(drop=True)

    return merged_df


# Quick schema self-check (no heavy compute).
candidate_outputs_df = load_all_model_outputs(CONFIG)
print("Combined output table shape:", candidate_outputs_df.shape)
display(
    candidate_outputs_df.groupby("model_name").agg(
        rows=("row_id", "count"),
        prompts=("prompt_id", "nunique"),
        acc_nan_ratio=("acc", lambda x: float(np.mean(np.isnan(x)))),
    )
)


In [ ]:
def resolve_device(config: AnalysisConfig) -> torch.device:
    """Resolve execution device based on config and runtime availability.

    Args:
        config: Global analysis configuration.

    Returns:
        Torch device used for teacher-forced scoring.
    """

    if config.device is not None:
        return torch.device(config.device)
    return torch.device("cuda" if torch.cuda.is_available() else "cpu")


def load_causal_lm_and_tokenizer(model_path: str | Path, device: torch.device):
    """Load HF causal LM + tokenizer for logit diagnostics.

    Args:
        model_path: HF model id or local checkpoint path.
        device: Compute device.

    Returns:
        Tuple of (model, tokenizer).
    """

    dtype = torch.float16 if device.type == "cuda" else torch.float32

    tokenizer = AutoTokenizer.from_pretrained(
        str(model_path),
        trust_remote_code=True,
    )

    model = AutoModelForCausalLM.from_pretrained(
        str(model_path),
        torch_dtype=dtype,
        trust_remote_code=True,
        low_cpu_mem_usage=True,
    )
    model.to(device)
    model.eval()

    return model, tokenizer


def build_teacher_forced_tensors(
    tokenizer,
    prompt_text: str,
    response_text: str,
    max_sequence_length: int,
) -> Optional[Tuple[torch.Tensor, int, bool]]:
    """Tokenize prompt+response and return tensors for teacher-forced scoring.

    Args:
        tokenizer: HF tokenizer.
        prompt_text: Prompt string (model input before generation).
        response_text: Generated response string.
        max_sequence_length: Hard cap for total token length.

    Returns:
        Tuple `(full_ids, prompt_len, is_truncated)` or None when unusable.
    """

    prompt_ids = tokenizer(prompt_text, add_special_tokens=False).input_ids
    full_ids = tokenizer(prompt_text + response_text, add_special_tokens=False).input_ids

    if len(prompt_ids) == 0 or len(full_ids) <= len(prompt_ids):
        return None

    # Keep implementation simple and safe: skip overlong sequences instead of silent truncation,
    # because truncation can distort entropy and confidence diagnostics.
    if len(full_ids) > max_sequence_length:
        return None

    full_tensor = torch.tensor(full_ids, dtype=torch.long)
    return full_tensor, len(prompt_ids), False


def compute_token_statistics(
    logits_for_response: torch.Tensor,
    target_token_ids: torch.Tensor,
    hidden_for_response: torch.Tensor,
) -> Dict[str, float]:
    """Compute token-level uncertainty and confidence metrics from logits.

    Args:
        logits_for_response: Logits aligned to response token targets, shape [T, V].
        target_token_ids: Response token ids, shape [T].
        hidden_for_response: Hidden states aligned to logits, shape [T, H].

    Returns:
        Dictionary with aggregate metrics used in calibration analysis.
    """

    logits_fp32 = logits_for_response.to(torch.float32)
    log_probs = torch.log_softmax(logits_fp32, dim=-1)
    probs = torch.exp(log_probs)

    selected_log_probs = log_probs.gather(-1, target_token_ids.unsqueeze(-1)).squeeze(-1)
    nll = -selected_log_probs

    entropy = -(probs * log_probs).sum(dim=-1)

    top2_probs = torch.topk(probs, k=2, dim=-1).values
    top1_prob = top2_probs[:, 0]
    margin = top2_probs[:, 0] - top2_probs[:, 1]

    logit_norm = torch.linalg.norm(logits_fp32, dim=-1)
    hidden_norm = torch.linalg.norm(hidden_for_response.to(torch.float32), dim=-1)

    nll_mean = float(nll.mean().item())
    selected_logprob_mean = float(selected_log_probs.mean().item())

    return {
        "token_count": int(target_token_ids.numel()),
        "nll_mean": nll_mean,
        "nll_std": float(nll.std(unbiased=False).item()),
        "ppl": float(math.exp(min(20.0, nll_mean))),
        "selected_logprob_mean": selected_logprob_mean,
        "selected_logprob_std": float(selected_log_probs.std(unbiased=False).item()),
        "confidence": float(math.exp(max(-20.0, selected_logprob_mean))),
        "entropy_mean": float(entropy.mean().item()),
        "entropy_std": float(entropy.std(unbiased=False).item()),
        "top1_prob_mean": float(top1_prob.mean().item()),
        "margin_mean": float(margin.mean().item()),
        "logit_norm_mean": float(logit_norm.mean().item()),
        "hidden_norm_mean": float(hidden_norm.mean().item()),
    }


def compute_teacher_forced_metrics_for_model(
    model_name: str,
    model_path: str | Path,
    rows_df: pd.DataFrame,
    config: AnalysisConfig,
) -> pd.DataFrame:
    """Run teacher-forced forward passes and compute uncertainty metrics.

    Args:
        model_name: Display name for reporting.
        model_path: HF model id or local path.
        rows_df: Candidate rows for this model.
        config: Global notebook configuration.

    Returns:
        DataFrame keyed by `row_id` with token-level aggregate metrics.
    """

    if rows_df.empty:
        raise ValueError(f"No rows to score for model {model_name}")

    device = resolve_device(config)
    print(f"[{model_name}] Loading model on device={device} from {model_path}")
    model, tokenizer = load_causal_lm_and_tokenizer(model_path=model_path, device=device)

    # Optional subsampling for fast iteration.
    scoring_df = rows_df.copy()
    if config.max_examples_for_teacher_forcing is not None and len(scoring_df) > config.max_examples_for_teacher_forcing:
        scoring_df = scoring_df.sample(
            n=config.max_examples_for_teacher_forcing,
            random_state=SEED,
            replace=False,
        ).reset_index(drop=True)

    metric_rows: List[Dict[str, object]] = []
    skipped_overlong = 0
    skipped_tokenization = 0

    with torch.no_grad():
        for row in tqdm(scoring_df.itertuples(index=False), total=len(scoring_df), desc=f"TF metrics: {model_name}"):
            prepared = build_teacher_forced_tensors(
                tokenizer=tokenizer,
                prompt_text=row.prompt,
                response_text=row.response,
                max_sequence_length=config.max_sequence_length,
            )
            if prepared is None:
                if len(tokenizer(row.prompt + row.response, add_special_tokens=False).input_ids) > config.max_sequence_length:
                    skipped_overlong += 1
                else:
                    skipped_tokenization += 1
                continue

            full_ids, prompt_len, _ = prepared

            input_ids = full_ids.unsqueeze(0).to(device)
            attention_mask = torch.ones_like(input_ids, device=device)

            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                output_hidden_states=True,
                use_cache=False,
            )

            # Align logits with response tokens under teacher forcing.
            # Predict token t using hidden state/logits at position t-1.
            response_start = prompt_len
            target_ids = input_ids[0, response_start:]
            logits = outputs.logits[0]
            last_hidden = outputs.hidden_states[-1][0]

            logits_for_response = logits[response_start - 1 : -1, :]
            hidden_for_response = last_hidden[response_start - 1 : -1, :]

            if logits_for_response.shape[0] != target_ids.shape[0]:
                skipped_tokenization += 1
                continue

            stat = compute_token_statistics(
                logits_for_response=logits_for_response,
                target_token_ids=target_ids,
                hidden_for_response=hidden_for_response,
            )
            stat.update(
                {
                    "row_id": row.row_id,
                    "model_name": model_name,
                    "scored_model_path": str(model_path),
                }
            )
            metric_rows.append(stat)

    # Explicit cleanup because models are large.
    del model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    metric_df = pd.DataFrame(metric_rows)
    print(
        f"[{model_name}] scored_rows={len(metric_df):,}, "
        f"skipped_overlong={skipped_overlong:,}, skipped_other={skipped_tokenization:,}"
    )

    return metric_df


In [ ]:
def compute_reliability_bins(
    frame: pd.DataFrame,
    confidence_col: str = "confidence",
    label_col: str = "acc",
    n_bins: int = 10,
) -> pd.DataFrame:
    """Build reliability table for one model.

    Args:
        frame: Model-specific dataframe.
        confidence_col: Confidence column in [0, 1].
        label_col: Binary correctness column.
        n_bins: Number of confidence bins.

    Returns:
        Reliability dataframe with per-bin confidence, accuracy, and gap.
    """

    usable = frame[[confidence_col, label_col]].dropna().copy()
    if usable.empty:
        return pd.DataFrame(columns=["bin_idx", "count", "confidence_mean", "accuracy_mean", "abs_gap"])

    usable[confidence_col] = usable[confidence_col].clip(0.0, 1.0)

    edges = np.linspace(0.0, 1.0, n_bins + 1)
    usable["bin_idx"] = pd.cut(
        usable[confidence_col],
        bins=edges,
        include_lowest=True,
        labels=False,
    )

    summary = (
        usable.groupby("bin_idx", dropna=False)
        .agg(
            count=(confidence_col, "count"),
            confidence_mean=(confidence_col, "mean"),
            accuracy_mean=(label_col, "mean"),
        )
        .reset_index()
    )
    summary["abs_gap"] = (summary["confidence_mean"] - summary["accuracy_mean"]).abs()
    return summary


def summarize_calibration(
    merged_df: pd.DataFrame,
    n_bins: int,
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """Compute calibration summaries and reliability tables by model.

    Args:
        merged_df: Output + metric dataframe.
        n_bins: Number of calibration bins.

    Returns:
        Tuple `(model_summary_df, reliability_bins_df)`.
    """

    model_rows: List[Dict[str, object]] = []
    bin_frames: List[pd.DataFrame] = []

    for model_name, group in merged_df.groupby("model_name"):
        usable = group[["confidence", "acc", "entropy_mean", "selected_logprob_mean"]].dropna()
        if usable.empty:
            continue

        bins = compute_reliability_bins(
            frame=group,
            confidence_col="confidence",
            label_col="acc",
            n_bins=n_bins,
        )
        if not bins.empty:
            bins["model_name"] = model_name
            bin_frames.append(bins)
            ece = float((bins["count"] * bins["abs_gap"]).sum() / max(bins["count"].sum(), 1.0))
        else:
            ece = float("nan")

        brier = float(np.mean(np.square(usable["confidence"].to_numpy() - usable["acc"].to_numpy())))

        model_rows.append(
            {
                "model_name": model_name,
                "samples": int(len(usable)),
                "ece": ece,
                "brier": brier,
                "confidence_mean": float(usable["confidence"].mean()),
                "acc_mean": float(usable["acc"].mean()),
                "entropy_mean": float(usable["entropy_mean"].mean()),
                "confidence_acc_spearman": float(usable["confidence"].corr(usable["acc"], method="spearman")),
                "entropy_acc_spearman": float(usable["entropy_mean"].corr(usable["acc"], method="spearman")),
            }
        )

    model_summary = pd.DataFrame(model_rows)
    reliability_bins = pd.concat(bin_frames, ignore_index=True) if bin_frames else pd.DataFrame()
    return model_summary, reliability_bins


def build_prompt_consistency_table(merged_df: pd.DataFrame) -> pd.DataFrame:
    """Aggregate per-prompt consistency diagnostics across n samples.

    Args:
        merged_df: Output + metric dataframe.

    Returns:
        Prompt-level table containing mean/best/worst and instability features.
    """

    rows: List[Dict[str, object]] = []

    for (model_name, prompt_id), group in merged_df.groupby(["model_name", "prompt_id"]):
        g = group.dropna(subset=["acc"]).copy()
        if g.empty:
            continue

        acc_best = float(g["acc"].max())
        acc_worst = float(g["acc"].min())
        acc_mean = float(g["acc"].mean())

        confidence_std = float(g["confidence"].std(ddof=0)) if "confidence" in g else float("nan")
        entropy_std = float(g["entropy_mean"].std(ddof=0)) if "entropy_mean" in g else float("nan")

        correct = g[g["acc"] >= 0.5]
        wrong = g[g["acc"] < 0.5]

        if (not correct.empty) and (not wrong.empty):
            conf_gap = float(correct["confidence"].mean() - wrong["confidence"].mean())
            entropy_gap = float(wrong["entropy_mean"].mean() - correct["entropy_mean"].mean())
            confidence_inversion = float(conf_gap < 0.0)
        else:
            conf_gap = float("nan")
            entropy_gap = float("nan")
            confidence_inversion = float("nan")

        rows.append(
            {
                "model_name": model_name,
                "prompt_id": prompt_id,
                "n_samples": int(len(g)),
                "acc_mean": acc_mean,
                "acc_best": acc_best,
                "acc_worst": acc_worst,
                "mixed_outcome": float(acc_best > acc_worst),
                "confidence_std": confidence_std,
                "entropy_std": entropy_std,
                "confidence_gap_correct_minus_wrong": conf_gap,
                "entropy_gap_wrong_minus_correct": entropy_gap,
                "confidence_inversion": confidence_inversion,
            }
        )

    return pd.DataFrame(rows)


def summarize_consistency(prompt_df: pd.DataFrame) -> pd.DataFrame:
    """Summarize prompt-level consistency metrics by model.

    Args:
        prompt_df: Prompt-level consistency dataframe.

    Returns:
        Model-level summary dataframe.
    """

    rows: List[Dict[str, object]] = []

    for model_name, group in prompt_df.groupby("model_name"):
        rows.append(
            {
                "model_name": model_name,
                "prompts": int(len(group)),
                "acc/mean": float(group["acc_mean"].mean()),
                "acc/best": float(group["acc_best"].mean()),
                "acc/worst": float(group["acc_worst"].mean()),
                "mixed_prompt_rate": float(group["mixed_outcome"].mean()),
                "confidence_std_mean": float(group["confidence_std"].mean()),
                "entropy_std_mean": float(group["entropy_std"].mean()),
                "confidence_gap_mean": float(group["confidence_gap_correct_minus_wrong"].mean()),
                "entropy_gap_mean": float(group["entropy_gap_wrong_minus_correct"].mean()),
                "confidence_inversion_rate": float(group["confidence_inversion"].mean()),
            }
        )

    return pd.DataFrame(rows)


In [ ]:
def _extract_last_layer_tensors(model) -> Dict[str, torch.Tensor]:
    """Extract final layers that directly influence output logits.

    Args:
        model: HF causal LM.

    Returns:
        Dictionary containing `lm_head_weight` and `final_norm_weight`.
    """

    if not hasattr(model, "lm_head"):
        raise AttributeError("Model does not expose lm_head")

    if not hasattr(model, "model") or not hasattr(model.model, "norm"):
        raise AttributeError("Model does not expose model.norm")

    return {
        "lm_head_weight": model.lm_head.weight.detach().to(torch.float32).cpu().clone(),
        "final_norm_weight": model.model.norm.weight.detach().to(torch.float32).cpu().clone(),
    }


def load_last_layer_tensors(model_path: str | Path) -> Dict[str, torch.Tensor]:
    """Load only last-layer tensors by briefly instantiating the full model.

    Args:
        model_path: HF model id or local checkpoint path.

    Returns:
        Dictionary of last-layer tensors.
    """

    model = AutoModelForCausalLM.from_pretrained(
        str(model_path),
        torch_dtype=torch.float16,
        trust_remote_code=True,
        low_cpu_mem_usage=True,
        device_map="cpu",
    )
    tensors = _extract_last_layer_tensors(model)

    del model
    gc.collect()

    return tensors


def compute_delta_metrics(
    base_tensor: torch.Tensor,
    target_tensor: torch.Tensor,
    tau_rms_factor: float = 0.1,
) -> Dict[str, float]:
    """Compute drift metrics between base and target tensors.

    Args:
        base_tensor: Reference tensor.
        target_tensor: Target tensor.
        tau_rms_factor: Active threshold = factor * RMS(delta).

    Returns:
        Metric dictionary for drift size/alignment/sign behavior.
    """

    base = base_tensor.reshape(-1).to(torch.float32)
    target = target_tensor.reshape(-1).to(torch.float32)

    if base.shape != target.shape:
        raise ValueError("Tensor shape mismatch in delta metrics")

    delta = target - base

    base_norm = float(torch.linalg.norm(base).item())
    target_norm = float(torch.linalg.norm(target).item())
    delta_norm = float(torch.linalg.norm(delta).item())

    cosine = float(torch.dot(base, target).item() / max(base_norm * target_norm, 1e-12))
    cosine = max(min(cosine, 1.0), -1.0)

    rms_delta = math.sqrt(float(torch.mean(delta * delta).item()))
    tau = tau_rms_factor * rms_delta

    active = torch.abs(delta) > tau
    sign_flip = (torch.sign(base) * torch.sign(target)) < 0

    active_count = max(int(active.sum().item()), 1)
    active_sign_flip = float((active & sign_flip).sum().item() / active_count)
    sign_flip_ratio = float(sign_flip.float().mean().item())

    return {
        "base_norm": base_norm,
        "target_norm": target_norm,
        "delta_norm": delta_norm,
        "delta_norm_ratio": delta_norm / max(base_norm, 1e-12),
        "cosine": cosine,
        "delta_abs_mean": float(torch.mean(torch.abs(delta)).item()),
        "delta_std": float(torch.std(delta, unbiased=False).item()),
        "active_sign_flip_ratio": active_sign_flip,
        "global_sign_flip_ratio": sign_flip_ratio,
        "tau": tau,
    }


def run_last_layer_diagnostics(
    config: AnalysisConfig,
    reference_model_name: str,
) -> pd.DataFrame:
    """Compare each configured model to one reference model at last-layer level.

    Args:
        config: Global analysis configuration.
        reference_model_name: Display name used as baseline.

    Returns:
        DataFrame with lm_head and final_norm drift metrics.
    """

    run_map = {run.display_name: run for run in config.model_runs}
    if reference_model_name not in run_map:
        raise ValueError(f"Reference model {reference_model_name} not found in config")

    reference_run = run_map[reference_model_name]
    reference_tensors = load_last_layer_tensors(reference_run.model_path)

    rows: List[Dict[str, object]] = []

    for run in config.model_runs:
        target_tensors = load_last_layer_tensors(run.model_path)

        lm_metrics = compute_delta_metrics(
            base_tensor=reference_tensors["lm_head_weight"],
            target_tensor=target_tensors["lm_head_weight"],
            tau_rms_factor=config.tau_rms_factor,
        )
        norm_metrics = compute_delta_metrics(
            base_tensor=reference_tensors["final_norm_weight"],
            target_tensor=target_tensors["final_norm_weight"],
            tau_rms_factor=config.tau_rms_factor,
        )

        rows.append(
            {
                "model_name": run.display_name,
                "reference_model": reference_model_name,
                "lm_head_delta_norm_ratio": lm_metrics["delta_norm_ratio"],
                "lm_head_cosine": lm_metrics["cosine"],
                "lm_head_active_sign_flip_ratio": lm_metrics["active_sign_flip_ratio"],
                "lm_head_global_sign_flip_ratio": lm_metrics["global_sign_flip_ratio"],
                "final_norm_delta_norm_ratio": norm_metrics["delta_norm_ratio"],
                "final_norm_cosine": norm_metrics["cosine"],
                "final_norm_active_sign_flip_ratio": norm_metrics["active_sign_flip_ratio"],
                "final_norm_global_sign_flip_ratio": norm_metrics["global_sign_flip_ratio"],
            }
        )

        del target_tensors
        gc.collect()

    del reference_tensors
    gc.collect()

    return pd.DataFrame(rows)


In [ ]:
def plot_mean_best_worst(consistency_summary: pd.DataFrame, output_dir: Path) -> None:
    """Plot model-wise mean/best/worst accuracy bars.

    Args:
        consistency_summary: Model-level consistency summary.
        output_dir: Artifact directory.
    """

    if consistency_summary.empty:
        return

    plot_df = consistency_summary.copy()
    long_df = plot_df.melt(
        id_vars=["model_name"],
        value_vars=["acc/mean", "acc/best", "acc/worst"],
        var_name="metric",
        value_name="value",
    )

    plt.figure(figsize=(12, 6))
    sns.barplot(data=long_df, x="model_name", y="value", hue="metric")
    plt.ylim(0.0, 1.0)
    plt.title("Mean/Best/Worst Accuracy by Model")
    plt.xlabel("Model")
    plt.ylabel("Accuracy")
    plt.xticks(rotation=20, ha="right")
    plt.tight_layout()

    path = output_dir / "mean_best_worst_accuracy.png"
    plt.savefig(path, dpi=180)
    plt.show()
    print(f"Saved figure: {path}")


def plot_reliability_diagram(reliability_bins: pd.DataFrame, output_dir: Path) -> None:
    """Plot reliability curves (confidence vs empirical accuracy).

    Args:
        reliability_bins: Concatenated reliability-bin table.
        output_dir: Artifact directory.
    """

    if reliability_bins.empty:
        return

    plt.figure(figsize=(10, 8))
    plt.plot([0, 1], [0, 1], linestyle="--", color="black", label="Perfect calibration")

    for model_name, group in reliability_bins.groupby("model_name"):
        plt.plot(
            group["confidence_mean"],
            group["accuracy_mean"],
            marker="o",
            linewidth=2,
            label=model_name,
        )

    plt.xlim(0.0, 1.0)
    plt.ylim(0.0, 1.0)
    plt.xlabel("Mean confidence (bin)")
    plt.ylabel("Empirical accuracy (bin)")
    plt.title("Reliability Diagram")
    plt.legend()
    plt.tight_layout()

    path = output_dir / "reliability_diagram.png"
    plt.savefig(path, dpi=180)
    plt.show()
    print(f"Saved figure: {path}")


def plot_uncertainty_distributions(merged_df: pd.DataFrame, output_dir: Path) -> None:
    """Plot entropy/confidence distributions split by correctness.

    Args:
        merged_df: Output + metric dataframe.
        output_dir: Artifact directory.
    """

    usable = merged_df.dropna(subset=["acc", "entropy_mean", "confidence"]).copy()
    if usable.empty:
        return

    usable["is_correct"] = usable["acc"] >= 0.5

    fig, axes = plt.subplots(1, 2, figsize=(16, 6), constrained_layout=True)

    sns.boxplot(
        data=usable,
        x="model_name",
        y="entropy_mean",
        hue="is_correct",
        ax=axes[0],
    )
    axes[0].set_title("Entropy Distribution by Model and Correctness")
    axes[0].set_xlabel("Model")
    axes[0].set_ylabel("Mean token entropy")
    axes[0].tick_params(axis="x", rotation=20)

    sns.boxplot(
        data=usable,
        x="model_name",
        y="confidence",
        hue="is_correct",
        ax=axes[1],
    )
    axes[1].set_title("Confidence Distribution by Model and Correctness")
    axes[1].set_xlabel("Model")
    axes[1].set_ylabel("Self-confidence")
    axes[1].tick_params(axis="x", rotation=20)

    path = output_dir / "uncertainty_distributions.png"
    plt.savefig(path, dpi=180)
    plt.show()
    print(f"Saved figure: {path}")


def plot_prompt_instability(prompt_df: pd.DataFrame, output_dir: Path) -> None:
    """Scatter instability against worst-case accuracy at prompt level.

    Args:
        prompt_df: Prompt-level consistency dataframe.
        output_dir: Artifact directory.
    """

    if prompt_df.empty:
        return

    plt.figure(figsize=(10, 7))
    sns.scatterplot(
        data=prompt_df,
        x="confidence_std",
        y="acc_worst",
        hue="model_name",
        style="mixed_outcome",
        alpha=0.7,
    )
    plt.title("Prompt-Level Instability vs Worst@N")
    plt.xlabel("Confidence std across n samples")
    plt.ylabel("Prompt worst accuracy")
    plt.tight_layout()

    path = output_dir / "prompt_instability_vs_worst.png"
    plt.savefig(path, dpi=180)
    plt.show()
    print(f"Saved figure: {path}")


def plot_last_layer_drift(last_layer_df: pd.DataFrame, output_dir: Path) -> None:
    """Visualize last-layer drift metrics for each model.

    Args:
        last_layer_df: Last-layer diagnostic dataframe.
        output_dir: Artifact directory.
    """

    if last_layer_df.empty:
        return

    fig, axes = plt.subplots(1, 2, figsize=(16, 6), constrained_layout=True)

    sns.barplot(
        data=last_layer_df,
        x="model_name",
        y="lm_head_delta_norm_ratio",
        ax=axes[0],
    )
    axes[0].set_title("LM-Head Delta Norm Ratio vs Reference")
    axes[0].set_xlabel("Model")
    axes[0].set_ylabel("||Δ lm_head|| / ||lm_head_ref||")
    axes[0].tick_params(axis="x", rotation=20)

    sns.barplot(
        data=last_layer_df,
        x="model_name",
        y="final_norm_delta_norm_ratio",
        ax=axes[1],
    )
    axes[1].set_title("Final-Norm Delta Norm Ratio vs Reference")
    axes[1].set_xlabel("Model")
    axes[1].set_ylabel("||Δ final_norm|| / ||final_norm_ref||")
    axes[1].tick_params(axis="x", rotation=20)

    path = output_dir / "last_layer_drift.png"
    plt.savefig(path, dpi=180)
    plt.show()
    print(f"Saved figure: {path}")


In [ ]:
# --------------------------------------------------------------------------------------
# Stage A: Teacher-forced logit metrics per model (self-output mode).
# --------------------------------------------------------------------------------------

available_run_map = {run.display_name: run for run in CONFIG.model_runs}

token_metric_frames: List[pd.DataFrame] = []

for model_name, group in candidate_outputs_df.groupby("model_name"):
    run_cfg = available_run_map.get(model_name)
    if run_cfg is None:
        print(f"[SKIP] {model_name}: run config missing")
        continue

    try:
        metric_df = compute_teacher_forced_metrics_for_model(
            model_name=model_name,
            model_path=run_cfg.model_path,
            rows_df=group,
            config=CONFIG,
        )
        token_metric_frames.append(metric_df)
    except Exception as exc:
        print(f"[WARN] Teacher-forced scoring failed for {model_name}: {exc}")

if not token_metric_frames:
    raise RuntimeError("No teacher-forced metrics were generated. Check model paths and memory limits.")

token_metrics_df = pd.concat(token_metric_frames, ignore_index=True)

analysis_df = candidate_outputs_df.merge(
    token_metrics_df,
    on=["row_id", "model_name"],
    how="inner",
)

print("Teacher-forced metric table shape:", analysis_df.shape)
metric_csv = ARTIFACT_DIR / "teacher_forced_logit_metrics.csv"
analysis_df.to_csv(metric_csv, index=False)
print(f"Saved metrics: {metric_csv}")

display(
    analysis_df.groupby("model_name").agg(
        rows=("row_id", "count"),
        prompts=("prompt_id", "nunique"),
        acc_mean=("acc", "mean"),
        entropy_mean=("entropy_mean", "mean"),
        confidence_mean=("confidence", "mean"),
    )
)


In [ ]:
# --------------------------------------------------------------------------------------
# Stage B: Calibration + consistency summaries and visualizations.
# --------------------------------------------------------------------------------------

calibration_summary_df, reliability_bins_df = summarize_calibration(
    merged_df=analysis_df,
    n_bins=CONFIG.calibration_bins,
)

prompt_consistency_df = build_prompt_consistency_table(analysis_df)
consistency_summary_df = summarize_consistency(prompt_consistency_df)

calibration_csv = ARTIFACT_DIR / "calibration_summary.csv"
reliability_csv = ARTIFACT_DIR / "reliability_bins.csv"
prompt_csv = ARTIFACT_DIR / "prompt_consistency.csv"
consistency_csv = ARTIFACT_DIR / "consistency_summary.csv"

calibration_summary_df.to_csv(calibration_csv, index=False)
reliability_bins_df.to_csv(reliability_csv, index=False)
prompt_consistency_df.to_csv(prompt_csv, index=False)
consistency_summary_df.to_csv(consistency_csv, index=False)

print(f"Saved table: {calibration_csv}")
print(f"Saved table: {reliability_csv}")
print(f"Saved table: {prompt_csv}")
print(f"Saved table: {consistency_csv}")

print("\n[Calibration Summary]")
display(calibration_summary_df.sort_values("ece"))

print("\n[Consistency Summary]")
display(consistency_summary_df.sort_values("acc/worst", ascending=False))

plot_mean_best_worst(consistency_summary_df, ARTIFACT_DIR)
plot_reliability_diagram(reliability_bins_df, ARTIFACT_DIR)
plot_uncertainty_distributions(analysis_df, ARTIFACT_DIR)
plot_prompt_instability(prompt_consistency_df, ARTIFACT_DIR)


In [ ]:
# --------------------------------------------------------------------------------------
# Stage C: Last-layer drift diagnostics relative to a reference model.
# --------------------------------------------------------------------------------------

reference_model_name = CONFIG.model_runs[0].display_name
print(f"Using reference model for last-layer comparison: {reference_model_name}")

last_layer_df = run_last_layer_diagnostics(
    config=CONFIG,
    reference_model_name=reference_model_name,
)

last_layer_csv = ARTIFACT_DIR / "last_layer_drift_summary.csv"
last_layer_df.to_csv(last_layer_csv, index=False)
print(f"Saved table: {last_layer_csv}")

display(last_layer_df)
plot_last_layer_drift(last_layer_df, ARTIFACT_DIR)


In [ ]:
# --------------------------------------------------------------------------------------
# Final compact report for quick review.
# --------------------------------------------------------------------------------------

report_df = consistency_summary_df.merge(
    calibration_summary_df[["model_name", "ece", "brier", "confidence_acc_spearman", "entropy_acc_spearman"]],
    on="model_name",
    how="left",
)

# Risk score emphasizes behaviors linked to worst-case degradation:
# high mixed outcome + high inversion + weak calibration + low worst accuracy.
if not report_df.empty:
    report_df["risk_score"] = (
        report_df["mixed_prompt_rate"].rank(pct=True)
        + report_df["confidence_inversion_rate"].rank(pct=True)
        + report_df["ece"].rank(pct=True)
        + (1.0 - report_df["acc/worst"]).rank(pct=True)
    )

    report_path = ARTIFACT_DIR / "final_risk_report.csv"
    report_df.sort_values("risk_score", ascending=False).to_csv(report_path, index=False)
    print(f"Saved report: {report_path}")

    display(report_df.sort_values("risk_score", ascending=False))
else:
    print("Report table is empty. Check previous stages.")
